# Week 2 Assessment Mini-Project

Complete all TODOs below. Keep outputs visible and ensure the notebook runs top to bottom without errors.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

In [ ]:
df = pd.read_csv("m1-09-assessment.csv", parse_dates=["date"])
df = df.set_index("date")
df_copy= df.copy()
print(df.head())


## Part A: Core Data Handling (TODOs)

- Inspect structure with `info()`, `describe()`, and missing value counts.
- Clean `pm25` by coercing invalid strings to NaN.
- Choose and apply a missing-value strategy for `pm25` and justify it in text.

In [ ]:
#print(df.info())
#print(df.describe())

df["pm25"] = pd.to_numeric(df["pm25"], errors="coerce")
df['pm25'] = df['pm25'].interpolate(method='linear')
df.isna().sum()

# #I chose Linear Interpolation because air quality is a time-series variable that typically changes gradually.
# Removing rows would break the daily continuity needed for rolling averages, and using the global mean would ignore seasonal trends (e.g., winter vs. summer). 
# Interpolation estimates missing values based on neighboring days, which is more representative of real-world atmospheric conditions.

## Part B: Required Analysis (TODOs)

- Data quality analysis: city with highest % invalid/missing pm25.
- Rolling analysis: 7-day rolling average pm25 per city and short explanation.
- Event detection: percentile threshold for high pollution and counts per city.
- Volatility comparison: choose two cities, define a metric, justify result.
- Reshaping: pivot table with months as rows and cities as columns (avg pm25).

In [ ]:
# Identify the city with the highest percentage of invalid/missing pm25.
missing_pct = df_copy.groupby('city')['pm25'].apply(lambda x: x.isnull().mean() * 100)
highest_missing_city = missing_pct.idxmax()
print(f"The highest missing: {highest_missing_city} ({missing_pct.max():.2f}%)")

# Compute a 7-day rolling average of pm25 per city and explain why this helps interpretation.
df['pm25_rolling'] = df.groupby('city')['pm25'].rolling(window=7).mean().reset_index(0, drop=True)

# Define a percentile-based high-pollution threshold and count events per city.
threshold = df['pm25'].quantile(0.90)
high_pollution_events = df[df['pm25'] > threshold].groupby('city').size()
print("\nHigh Pollution cities:\n", high_pollution_events)

# Compare volatility between two cities using a metric you define and justify.
volatility = df.groupby('city')['pm25'].std()
print("\nSTD:\n", volatility)

# Create a month-by-city pivot table of average pm25.
pivot_table = df.pivot_table(values='pm25', index=df.index.month, columns='city', aggfunc='mean')
print("\n\n Pivot Table")
print(pivot_table)

## Part C: Aggregations (TODOs)

- Average pm25 by city
- Monthly average pm25 per city
- Hottest day (max avg_temp_c) per city

In [ ]:
# avg_pm25_by_city = ...
city_avg_pm25 = df.groupby('city')['pm25'].mean()
print("Average PM25 by city:")
print(city_avg_pm25)

# monthly_avg_pm25 = ...
monthly_city_avg = df.groupby(['city', df.index.month])['pm25'].mean()
print("\n\nMonthly average PM25 per city:\n")
print(monthly_city_avg)

# hottest_day_per_city = ...
hottest_day_per_city = df.loc[df.groupby('city')['avg_temp_c'].idxmax()]
print(hottest_day_per_city[['avg_temp_c']])

## Part D: Visualization (TODOs)

- Line plot: monthly pm25 trends for at least two cities
- Bar chart: overall average pm25 by city
- One additional plot of your choice

In [ ]:
# TODO: plotting section
import seaborn as sns

#1
plt.figure(figsize=(10, 5))
pivot_table[['London', 'Warsaw']].plot(kind='line', marker='o', ax=plt.gca())
plt.title('Monthly PM2.5 Trends: London vs Warsaw')
plt.xlabel('Month')
plt.grid(alpha=0.3)
plt.ylabel('Average PM25')
plt.legend(title='City')
plt.show()

#2
plt.figure(figsize=(12, 7))
sns.set_style("whitegrid")
city_avg = df.groupby('city')['pm25'].mean().sort_values(ascending=False)
colors = sns.color_palette("Reds_r", len(city_avg))
bars = plt.bar(city_avg.index, city_avg.values, color=colors, edgecolor='black',  alpha=0.9)
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 0.3, round(yval, 2), 
             ha='center', va='bottom', fontweight='bold', fontsize=11)

plt.title('Average PM25 Levels by City (Jan - Jun 2023)', fontsize=15, pad=20)
plt.ylabel('PM2.5 Concentration', fontsize=12)
plt.xlabel('European Cities', fontsize=12)
plt.ylim(0, city_avg.max() + 5) 
plt.xticks(rotation=0) 
plt.tight_layout()
plt.show()

#3
plt.figure(figsize=(10, 6))
sns.boxplot(x='city', y='pm25', data=df)
plt.title('Distribution of PM2.5 Levels per City')
plt.xlabel('City')
plt.ylabel('PM2.5 Concentration')
plt.show()


## Part E: Interpretation Questions (TODOs)

Write short answers here:

1. Which city shows the most persistent high pm25 levels, and what evidence supports that?
2. How does missing or invalid data affect your confidence in the results?
3. Does temperature appear related to pm25 in your analysis? Explain briefly.
4. What is one limitation of using daily averages for air-quality policy decisions?
5. If you had one more dataset to improve this analysis, what would it be and why?

# Which city shows the most persistent high pm25 levels, and what evidence supports that? 

# Based on the analysis, Warsaw shows the most persistent high PM25 levelsHigh PM25 levels in Warsaw are primarily driven by a combination of, "low-stack emissions" (residential heating), 
# heavy road traffic, and unfavorable winter weather conditions that trap pollutants
#---------------------------------------------------------------------------------------------------------------------------------------------------------------

# #How does missing or invalid data affect your confidence in the results?

# Missing or invalid data reduces the statistical confidence in our results, as it can hide extreme pollution events or skew monthly averages. 
# If critical days (e.g., during a heatwave or a cold snap) are missing, our calculated "mean" might represent a cleaner or dirtier reality than what actually occurred.
#---------------------------------------------------------------------------------------------------------------------------------------------------------------

#Does temperature appear related to pm25 in your analysis? Explain briefly.

# Yes, there appears to be an inverse relationship. As the avg_temp_c increases from January to June, PM25 levels drop significantly across all cities. 
# This suggests that colder temperatures likely coincide with higher heating demands and atmospheric inversions that trap pollutants near the ground.

#-----------------------------------------------------------------------------------------------------------------------------------------------------------------

#What is one limitation of using daily averages for air-quality policy decisions?

# Daily averages smooth out short-term "spikes" in pollution.
# For example, a dangerous surge in PM25 during morning rush hour might be masked by lower levels at night, leading policymakers to underestimate the acute health risks posed to commuters.

#------------------------------------------------------------------------------------------------------------------------------------------------------

# If you had one more dataset to improve this analysis, what would it be and why?

# I would definitely add a dataset on wind speed and direction (Wind Data) to this analysis. 
# This is because pollution does not depend only on temperature, but also on atmospheric movement. 
# For example, even if the sources of pollution in a city remain the same, a strong wind can disperse accumulated $PM_{2.5}$ particles in the air within seconds. 
# If we had wind data, we could more accurately determine whether high pollution levels are caused by atmospheric stagnation or by the emergence of a new pollution source.




